# 02 · Datasets: two pools for training

## Setup

In [1]:
import os, sys

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # 1. Get the code: your reasoning branch from GitHub
    if not os.path.exists("/content/BN4601A"):
        !git clone -b reasoning https://github.com/likingood16/BN4601A.git /content/BN4601A
    else:
        !git -C /content/BN4601A pull
    %cd /content/BN4601A

    # 2. Libraries Colab doesn't have (Colab already has torch, pandas, numpy, scipy, matplotlib, pillow, tqdm, joblib)
    !pip install -q -U transformers accelerate bitsandbytes   # load MedGemma in 4-bit
    !pip install -q -U peft trl                               # LoRA fine-tuning (task 6, optional)
    !pip install -q -U datasets kagglehub                     # HF datasets (SynthVision, MedQA, MultiCaRe) + KTAS download
    !pip install -q scikit-learn==1.9.1                       # Caleb's saved .joblib models need this exact version

    # 3. KTAS data (not on GitHub): same place as on the laptop
    if not os.path.exists("vitals/data/raw/ktas_triage.csv"):
        import kagglehub, shutil
        os.makedirs("vitals/data/raw", exist_ok=True)
        shutil.copy(kagglehub.dataset_download("ilkeryildiz/emergency-service-triage-application") + "/data.csv",
                    "vitals/data/raw/ktas_triage.csv")

    # 4. Hugging Face login (MedGemma is gated). getpass hides the token and keeps it out of the notebook
    from huggingface_hub import login, whoami
    try:
        whoami()
    except Exception:
        from getpass import getpass
        login(token=getpass("Hugging Face token: "))
else:
    %cd ../..

# Caleb's code (VitalsModel, ktas_data) importable from here on
sys.path.insert(0, "vitals/src")
print("IN_COLAB:", IN_COLAB, "| cwd:", os.getcwd(), "| KTAS data:", os.path.exists("vitals/data/raw/ktas_triage.csv"))

C:\Users\likin\Documents\BN4601A\group project\BN4601A
IN_COLAB: False | cwd: C:\Users\likin\Documents\BN4601A\group project\BN4601A | KTAS data: True


## KTAS: load + clean (Caleb's code)

In [2]:
import pandas as pd
import ktas_data as kd        # Caleb's file: vitals/src/ktas_data.py

raw = kd.load_raw()           # the Kaggle file, exactly as downloaded
ktas, log = kd.clean(raw)     # fixes typos, merges duplicate visits, turns codes into numbers

print("raw rows:", len(raw), "-> clean rows:", len(ktas))
log                           # what cleaning changed, step by step

raw rows: 1267 -> clean rows: 1262


,step,column,n_rows,detail
0,rename,all,1267,columns renamed to snake_case
1,placeholder -> missing,chief_complaint,5,'??'
2,placeholder -> missing,nrs_pain,556,'#BOŞ!'
3,placeholder -> missing,sbp,25,'??'
4,placeholder -> missing,dbp,29,'??'
5,placeholder -> missing,hr,20,'??'
6,placeholder -> missing,rr,22,'??'
7,placeholder -> missing,temp_c,18,'??'
8,placeholder -> missing,spo2,9,'??'
9,trailing dot,dbp,1,e.g. '36.' read as 36.0


## KTAS: split (same patients as Caleb, seed 42)

In [3]:
splits = kd.make_splits(ktas, seed=42)          # visit_id -> train / val / test
ktas = ktas.merge(splits, on="visit_id")
ktas["urgent"] = (ktas["ktas_expert"] <= 2).astype(int)   # KTAS 1-2 = urgent, same as Caleb

table = pd.crosstab(ktas["split"], ktas["urgent"], margins=True)
table.columns = ["not urgent", "urgent", "total"]
table

,not urgent,urgent,total
split,,,
test,153,37,190
train,706,176,882
val,158,32,190
All,1017,245,1262


In [4]:
# Guard: the test set must be Caleb's 190 patients (37 urgent), or comparisons with his table are invalid
test = ktas[ktas["split"] == "test"]
assert len(test) == 190 and test["urgent"].sum() == 37, "split differs from Caleb's"

# If Caleb's saved split exists on this machine, check it's the exact same visit_ids
from pathlib import Path
caleb_split = Path("vitals/data/processed/splits.csv")
if caleb_split.exists():
    caleb_test = set(pd.read_csv(caleb_split).query("split == 'test'")["visit_id"])
    assert caleb_test == set(test["visit_id"]), "different test patients"
    print("same 190 test visit_ids as Caleb's splits.csv")
else:
    print("splits.csv not here (e.g. on Colab); counts match Caleb's table")

same 190 test visit_ids as Caleb's splits.csv


## KTAS: one patient + columns we must never use

In [5]:
print("Never use as model input:", kd.LEAKAGE_COLS)

# What patients look like: allowed info only
allowed = ["age", "sex_label", "arrival_mode_label", "injury_label", "chief_complaint",
           "mental_label", "nrs_pain", "sbp", "dbp", "hr", "rr", "temp_c", "spo2"]
ktas.loc[ktas["split"] == "train", allowed + ["ktas_expert", "urgent"]].head(5)

Never use as model input: ['ktas_nurse', 'diagnosis', 'disposition', 'error_group', 'los_min', 'ktas_duration_min', 'mistriage']


,age,sex_label,arrival_mode_label,injury_label,chief_complaint,mental_label,nrs_pain,sbp,dbp,hr,rr,temp_c,spo2,ktas_expert,urgent
4,58,male,private_vehicle,no,"distension, abd",alert,3.0,91.0,67.0,93.0,18.0,36.5,NaN,5,0
5,54,female,private_ambulance,no,fever,alert,3.0,140.0,90.0,94.0,20.0,38.1,98.0,4,0
6,49,male,private_vehicle,no,With chest discomfort,alert,3.0,110.0,70.0,70.0,20.0,36.2,98.0,3,0
8,32,male,private_vehicle,no,LBP - Low back pain,alert,3.0,140.0,75.0,91.0,20.0,36.6,NaN,5,0
9,38,female,private_vehicle,no,"Eczema, Eyelid",alert,3.0,130.0,80.0,80.0,20.0,36.3,97.0,5,0


## NHAMCS: load US ED visits (Caleb's code)

In [6]:
import nhamcs                          # Caleb's file: vitals/src/nhamcs.py

nh_raw = nhamcs.load()                 # CDC file, only the columns Caleb needs
nh, nh_log = nhamcs.harmonize(nh_raw)  # adults, triage 1-5, KTAS-style column names and units
nh_log

,step,n
0,NHAMCS 2022 ED visits,16025
1,aged 16 or over,12999
2,triage level 1-5 recorded,8412
3,implausible values set to missing: 56 values,8412


## NHAMCS: reason for visit as complaint text

In [7]:
# Caleb's table has no complaint text. NHAMCS stores up to 3 "reasons for visit" as codes (e.g. 10500);
# the Stata file has a dictionary code -> text. Codes below 0 mean blank/unknown.
with pd.io.stata.StataReader(nhamcs.RAW) as reader:
    code_to_text = reader.value_labels()["RFVF"]

rfv = pd.read_stata(nhamcs.RAW, columns=["AGE", "IMMEDR", "RFV1", "RFV2", "RFV3"], convert_categoricals=False)
rfv = rfv[(rfv.AGE >= 16) & rfv.IMMEDR.between(1, 5)].reset_index(drop=True)   # same rows as harmonize()
assert len(rfv) == len(nh) and (rfv.AGE.values == nh.age.values).all()          # rows line up

def complaint(row):
    reasons = [code_to_text.get(c) for c in (row.RFV1, row.RFV2, row.RFV3) if c > 0]
    return "; ".join(r for r in reasons if r) or None

nh["chief_complaint"] = rfv.apply(complaint, axis=1)
nh["chief_complaint"].value_counts().head(10)

chief_complaint
Abdominal pain, cramps, spasms, NOS                      209
Chest pain                                               144
Shortness of breath                                       92
Back pain, ache, soreness, discomfort                     85
Oth symptoms/problems relat to psycho...                  69
Headache, pain in head                                    53
Accident, NOS                                             52
Laceration/cut of upper extremity                         48
Leg pain, ache, soreness, discomfort                      48
Abdominal pain, cramps, spasms, NOS; Nausea; Vomiting     41
Name: count, dtype: int64

## NHAMCS: split 70 / 15 / 15 (seed 42, same urgent share in each)

In [8]:
from sklearn.model_selection import train_test_split

train_idx, rest_idx = train_test_split(nh.index, test_size=0.30, stratify=nh.triage_level, random_state=42)
val_idx, test_idx = train_test_split(rest_idx, test_size=0.50, stratify=nh.triage_level[rest_idx], random_state=42)
nh["split"] = "train"
nh.loc[val_idx, "split"] = "val"
nh.loc[test_idx, "split"] = "test"

table = pd.crosstab(nh["split"], nh["urgent"], margins=True)
table.columns = ["not urgent", "urgent", "total"]
table

,not urgent,urgent,total
split,,,
test,1027,235,1262
train,4793,1095,5888
val,1028,234,1262
All,6848,1564,8412


## Pool 1: KTAS + NHAMCS in one table

In [9]:
# Same columns for both sources. Only information known at the triage desk.
ambulance_codes = [2, 4]   # KTAS arrival_mode: public / private ambulance

k = pd.DataFrame({
    "source": "ktas", "id": "ktas_" + ktas["visit_id"].astype(str), "split": ktas["split"],
    "age": ktas["age"], "sex": ktas["sex_label"],
    "ambulance": ktas["arrival_mode"].isin(ambulance_codes).astype(float),
    "injured": ktas["injured"].astype(float), "mental": ktas["mental_label"],
    "chief_complaint": ktas["chief_complaint"], "nrs_pain": ktas["nrs_pain"],
    "sbp": ktas["sbp"], "dbp": ktas["dbp"], "hr": ktas["hr"], "rr": ktas["rr"],
    "temp_c": ktas["temp_c"], "spo2": ktas["spo2"],
    "triage_level": ktas["ktas_expert"], "urgent": ktas["urgent"],
})
n = pd.DataFrame({
    "source": "nhamcs", "id": "nhamcs_" + nh.index.astype(str), "split": nh["split"],
    "age": nh["age"], "sex": nh["is_male"].map({1.0: "male", 0.0: "female"}),
    "ambulance": nh["ambulance"], "injured": nh["injured"], "mental": None,   # NHAMCS has no AVPU
    "chief_complaint": nh["chief_complaint"], "nrs_pain": nh["nrs_pain"],
    "sbp": nh["sbp"], "dbp": nh["dbp"], "hr": nh["hr"], "rr": nh["rr"],
    "temp_c": nh["temp_c"], "spo2": nh["spo2"],
    "triage_level": nh["triage_level"], "urgent": nh["urgent"],
})
pool1 = pd.concat([k, n], ignore_index=True)

table = pd.crosstab([pool1["source"], pool1["split"]], pool1["urgent"], margins=True)
table.columns = ["not urgent", "urgent", "total"]
table

not urgent  urgent  total
source split                           
ktas   test          153      37    190
       train         706     176    882
       val           158      32    190
nhamcs test         1027     235   1262
       train        4793    1095   5888
       val          1028     234   1262
All                 7865    1809   9674

In [10]:
# Checks before saving
assert not set(kd.LEAKAGE_COLS) & set(pool1.columns), "leakage column in Pool 1"
assert pool1["id"].is_unique
assert (pool1.query("source == 'ktas' and split == 'test'")["urgent"].agg(["size", "sum"]) == [190, 37]).all()

out = Path("reasoning/data/processed")
out.mkdir(parents=True, exist_ok=True)
pool1.to_parquet(out / "pool1.parquet", index=False)
print("saved", len(pool1), "patients ->", out / "pool1.parquet")
pool1.isna().mean().mul(100).round(1).rename("% missing").to_frame().T

saved 9674 patients -> reasoning\data\processed\pool1.parquet


,source,id,split,age,sex,ambulance,injured,mental,chief_complaint,nrs_pain,sbp,dbp,hr,rr,temp_c,spo2,triage_level,urgent
% missing,0.0,0.0,0.0,0.0,0.0,1.6,5.4,87.0,0.1,22.1,1.4,1.5,1.8,2.1,2.7,8.8,0.0,0.0


# Pool 2: TEXT (trains the LoRA adapter): ALL data, no sampling

## Settings + helpers

In [11]:
import json, re, glob, random, hashlib, gc
from sklearn.model_selection import train_test_split

RAW = Path("reasoning/data/raw")
SEED = 42
# No sampling here: every usable example goes into Pool 2.
# How many to TRAIN on is chosen later in the LoRA notebook (MAX_TRAIN_EXAMPLES).

# Words that make an example "emergency" (ED-like). Everything else is "general".
EMERGENCY_WORDS = [
    "emergency", "chest pain", "shortness of breath", "dyspnea", "breathless", "collapse", "syncope",
    "faint", "unconscious", "unresponsive", "seizure", "stroke", "slurred", "weakness on one side",
    "bleeding", "hemorrhage", "haemorrhage", "trauma", "injury", "fracture", "fall", "accident",
    "burn", "overdose", "poisoning", "anaphylaxis", "allergic reaction", "sepsis", "fever",
    "hypotension", "tachycardia", "cardiac arrest", "myocardial infarction", "heart attack",
    "acute abdomen", "severe pain", "vomiting blood", "hematemesis", "suicid", "confusion",
]
EMERGENCY_RE = re.compile(r"\b(" + "|".join(map(re.escape, EMERGENCY_WORDS)) + ")", re.I)

def scope(text):
    return "emergency" if EMERGENCY_RE.search(text or "") else "general"

def chat(user, assistant, system=None):
    """One training example in chat format: the same format ask() uses."""
    msgs = [{"role": "system", "content": system}] if system else []
    return msgs + [{"role": "user", "content": user}, {"role": "assistant", "content": assistant}]

def norm(text):
    return re.sub(r"[^a-z0-9]+", " ", (text or "").lower()).strip()

def as_text(messages):
    return " ".join(m["content"] or "" for m in messages)

## MedQA: 200-question benchmark (held out, never trained on)

In [12]:
def read_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8")]

def medqa_prompt(q):
    options = "\n".join(f"{k}. {v}" for k, v in sorted(q["options"].items()))
    return f"{q['question']}\n\n{options}\n\nAnswer with the letter of the best option."

medqa_test = read_jsonl(RAW / "medqa/phrases_no_exclude_test.jsonl")
bench_idx = random.Random(SEED).sample(range(len(medqa_test)), 200)
bench_rows = [{"id": f"medqa_test_{i}", "prompt": medqa_prompt(medqa_test[i]),
               "answer": medqa_test[i]["answer_idx"], "answer_text": medqa_test[i]["answer"]} for i in bench_idx]

# One compiled pattern of all 200 questions (normalised start): finds hidden copies fast
BENCH_RE = re.compile("|".join(re.escape(norm(medqa_test[i]["question"])[:150]) for i in bench_idx))

def contains_bench(messages):
    return bool(BENCH_RE.search(norm(as_text(messages))))

print(len(bench_rows), "benchmark questions; example answer:", bench_rows[0]["answer"], "-", bench_rows[0]["answer_text"])

200 benchmark questions; example answer: A - Chikungunya


## SynthVision: case Q&A + reports (MultiCaRe-based records dropped)

In [13]:
sv = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(str(RAW / "synthvision/filtered_*.parquet")))])
print("all records by source:", sv["source"].value_counts().to_dict())
sv = sv[sv["source"] != "multicare"].reset_index(drop=True)   # leakage guard: MultiCaRe = our pipeline test set
print("after dropping MultiCaRe:", len(sv))

SV_SYSTEM = "You are a clinical assistant. Answer using the imaging report provided; do not invent findings."

def sv_example(row):
    turns = json.loads(row["conversations"])
    msgs = [{"role": "system", "content": SV_SYSTEM}]
    for i, t in enumerate(turns):
        role = "user" if t["from"] == "human" else "assistant"
        text = t["value"]
        if i == 0:   # images aren't included, so the report text is the context
            text = f"Imaging report:\n{row['report']}\n\n{text}"
        msgs.append({"role": role, "content": text})
    return msgs

sv_all = pd.DataFrame({"id": "synthvision_" + sv["id"].astype(str), "source": "synthvision",
                       "scope": sv["report"].map(scope), "messages": sv.apply(sv_example, axis=1)})
del sv; gc.collect()
sv_all["scope"].value_counts()

all records by source: {'roco': 30740, 'multicare': 22855, 'pathvqa': 1640, 'vqarad': 124}
after dropping MultiCaRe: 32504


scope
general      23344
emergency     9160
Name: count, dtype: int64

## MedDialog: patient question → doctor answer (train + validation files)

In [14]:
parts = []
for name in ["train", "validation"]:
    d = pd.DataFrame(json.load(open(RAW / f"meddialog/{name}.json", encoding="utf-8")))
    d["id"] = f"meddialog_{name}_" + d.index.astype(str)
    parts.append(d)
md_raw = pd.concat(parts, ignore_index=True)
print("MedDialog total:", len(md_raw))
md_raw = md_raw[md_raw["doctor_response"].str.len().between(100, 2000)]      # drop empty / huge answers
print("answer 100-2,000 characters:", len(md_raw))

md_all = pd.DataFrame({"id": md_raw["id"].values, "source": "meddialog",
                       "scope": md_raw["patient_message"].map(scope).values,
                       "messages": [chat(q, a) for q, a in zip(md_raw["patient_message"], md_raw["doctor_response"])]})
del md_raw, parts; gc.collect()
md_all["scope"].value_counts()

MedDialog total: 251731
answer 100-2,000 characters: 248399


scope
general      185947
emergency     62452
Name: count, dtype: int64

## Medical reasoning: one question → reasoned answer (both files)

In [15]:
parts = []
for f in sorted(glob.glob(str(RAW / "reasoning_sft/data/*.parquet"))):   # one file at a time to save memory
    rs = pd.read_parquet(f)
    question = rs["messages"].map(lambda m: m[0]["content"])
    answer = rs["messages"].map(lambda m: m[-1]["content"] or "")
    keep = ((rs["messages"].map(len) == 2)                        # single question + answer
            & answer.str.len().between(200, 3000)                 # not too long for training
            & ~question.str.contains("summarize the following", case=False))   # skip summarising chores
    tag = Path(f).stem.split("-")[1]                               # 00000 / 00001
    parts.append(pd.DataFrame({"id": [f"reasoning_{tag}_{i}" for i in rs.index[keep]], "source": "reasoning",
                               "scope": question[keep].map(scope).values,
                               "messages": [chat(q, a) for q, a in zip(question[keep], answer[keep])]}))
    print(Path(f).name, "->", int(keep.sum()), "usable of", len(rs))
    del rs, question, answer; gc.collect()
rs_all = pd.concat(parts, ignore_index=True)
rs_all["scope"].value_counts()

train-00000-of-00002.parquet -> 51355 usable of 65864


train-00001-of-00002.parquet -> 51855 usable of 65864


scope
general      96799
emergency     6411
Name: count, dtype: int64

## MedQA train: exam question → letter + answer

In [16]:
mq = pd.DataFrame(read_jsonl(RAW / "medqa/phrases_no_exclude_train.jsonl"))
mq_all = pd.DataFrame({"id": "medqa_train_" + mq.index.astype(str), "source": "medqa",
                       "scope": mq["question"].map(scope),
                       "messages": [chat(medqa_prompt(r), f"{r['answer_idx']}. {r['answer']}") for _, r in mq.iterrows()]})
mq_all["scope"].value_counts()

scope
general      5114
emergency    5064
Name: count, dtype: int64

## Pool 2: combine, remove benchmark copies + duplicates, split 90 / 5 / 5

In [17]:
pool2 = pd.concat([sv_all, md_all, rs_all, mq_all], ignore_index=True)
del sv_all, md_all, rs_all, mq_all; gc.collect()
print("combined:", len(pool2))

# Leakage guard: drop any example that contains a MedQA benchmark question (about 2 minutes)
leaked = pool2["messages"].map(contains_bench)
print("examples containing a benchmark question (dropped):", int(leaked.sum()),
      pool2.loc[leaked, "source"].value_counts().to_dict())
pool2 = pool2[~leaked].reset_index(drop=True)

# Leakage guard: no exact duplicates (compare a short fingerprint of the normalised text)
fingerprint = pool2["messages"].map(lambda m: hashlib.md5(norm(as_text(m)).encode()).hexdigest())
dup = fingerprint.duplicated()
print("exact duplicates dropped:", int(dup.sum()), pool2.loc[dup, "source"].value_counts().to_dict())
pool2 = pool2[~dup].reset_index(drop=True)

# Split per source x scope, so every split has the same mix
pool2["split"] = "train"
for _, g in pool2.groupby(["source", "scope"]):
    _, rest = train_test_split(g.index, test_size=0.10, random_state=SEED)
    val, test = train_test_split(rest, test_size=0.50, random_state=SEED)
    pool2.loc[val, "split"] = "val"
    pool2.loc[test, "split"] = "test"

pd.crosstab([pool2["source"], pool2["scope"]], pool2["split"], margins=True)[["train", "val", "test", "All"]]

combined: 394291


examples containing a benchmark question (dropped): 2 {'medqa': 2}


exact duplicates dropped: 7864 {'meddialog': 7550, 'reasoning': 302, 'synthvision': 12}


split                   train    val   test     All
source      scope                                  
meddialog   emergency   55018   3057   3057   61132
            general    161745   8986   8986  179717
medqa       emergency    4556    253    254    5063
            general      4601    256    256    5113
reasoning   emergency    5769    320    321    6410
            general     86848   4825   4825   96498
synthvision emergency    8240    458    458    9156
            general     21002   1167   1167   23336
All                    347779  19322  19324  386425

## Options for the LoRA experiments (used in the LoRA notebook)

In [18]:
# TRAIN_SCOPE: "mixed" (all), "emergency", "general"; for the two-stage arm: "general" then "emergency"
def lora_train_set(pool, train_scope="mixed"):
    train = pool[pool["split"] == "train"]
    return train if train_scope == "mixed" else train[train["scope"] == train_scope]

for s in ["mixed", "emergency", "general"]:
    print(f"{s:10s} -> {len(lora_train_set(pool2, s)):7,d} training examples available")

mixed      -> 347,779 training examples available
emergency  ->  73,583 training examples available
general    -> 274,196 training examples available


## Save Pool 2 + benchmark

In [19]:
out = Path("reasoning/data/processed")
for split in ["train", "val", "test"]:
    pool2[pool2["split"] == split].to_json(out / f"pool2_{split}.jsonl", orient="records", lines=True, force_ascii=False)
pd.DataFrame(bench_rows).to_json(out / "medqa_bench200.jsonl", orient="records", lines=True, force_ascii=False)
for f in sorted(out.iterdir()):
    print(f"{f.stat().st_size / 1e6:8.1f} MB  {f.name}")

# One example, as the model will see it
for m in pool2.iloc[0]["messages"]:
    print(f"--- {m['role']}:\n{m['content'][:400]}")

     0.2 MB  medqa_bench200.jsonl
     0.3 MB  pool1.parquet
    28.2 MB  pool2_test.jsonl
   505.8 MB  pool2_train.jsonl
    28.0 MB  pool2_val.jsonl
--- system:
You are a clinical assistant. Answer using the imaging report provided; do not invent findings.
--- user:
Imaging report:
This cranio-caudal (CC) view mammogram of the left breast demonstrates the successful placement of a localization wire. The wire enters from the inferior aspect, traversing the breast tissue with its tip terminating at a focal density in the upper outer quadrant, which has been manually circled to indicate the target lesion. The surrounding breast parenchyma is heterogeneously dens
--- assistant:
The image shows a localization wire, often called a hookwire, which has been inserted into the breast tissue to mark a specific area for the surgeon.
--- user:
Why is this wire necessary if the lump can be seen on the mammogram?
--- assistant:
While the lump is visible on the mammogram, it may be non-palpable, mea

# Checks + held-out test sets

## Leakage checks on both pools

In [20]:
checks = {}

# 1. KTAS test = Caleb's 190 patients, 37 urgent
kt = pool1.query("source == 'ktas' and split == 'test'")
checks["KTAS test is Caleb's 190 / 37"] = len(kt) == 190 and kt["urgent"].sum() == 37

# 2. No forbidden (after-triage) columns in Pool 1
checks["no LEAKAGE_COLS in Pool 1"] = not set(kd.LEAKAGE_COLS) & set(pool1.columns)

# 3. Each patient / example in exactly one split
checks["Pool 1 ids unique"] = pool1["id"].is_unique
checks["Pool 2 ids unique"] = pool2["id"].is_unique

# 4. No MultiCaRe-based SynthVision record left (MultiCaRe cases are our pipeline test set)
checks["no MultiCaRe records in Pool 2"] = not pool2["id"].str.startswith("synthvision_multicare").any()

# 5. No MedQA benchmark question left in Pool 2 (already removed above; re-checked on val + test only, fast)
checks["no benchmark copies in Pool 2 val/test"] = not pool2.loc[pool2["split"] != "train", "messages"].map(contains_bench).any()

pd.Series(checks, name="passed").to_frame()

,passed
KTAS test is Caleb's 190 / 37,True
no LEAKAGE_COLS in Pool 1,True
Pool 1 ids unique,True
Pool 2 ids unique,True
no MultiCaRe records in Pool 2,True
no benchmark copies in Pool 2 val/test,True


In [21]:
assert all(checks.values()), "a leakage check failed"
del pool2; gc.collect()   # free memory before loading MultiCaRe
print("all checks passed")

all checks passed


## MultiCaRe ED cases: full-pipeline test set (never trained on)

In [22]:
sys.path.insert(0, "multicare_filter/src")
import multicare_filter as mf          # Caleb's file: multicare_filter/src/multicare_filter.py
from huggingface_hub import hf_hub_download

# Caleb's loader skips the download if the dataset folder exists but the file doesn't: fetch the file first
hf_hub_download("OpenMed/multicare-cases", "cases.parquet", repo_type="dataset")
cases = mf.annotate_cases(mf.load_cases())   # adds: came to ED?, which vitals are mentioned, department guess

# Caleb's case-level rules (no image rules: Model 3 only needs the case text)
steps = [("all cases", pd.Series(True, index=cases.index)),
         ("age >= 16", cases["age"] >= 16),
         ("presented to the emergency department", cases["ed_presentation"]),
         (">= 1 measured vital sign in the text", cases["n_vitals"] >= 1)]
keep = pd.Series(True, index=cases.index)
funnel = []
for name, mask in steps:
    keep &= mask.fillna(False)
    funnel.append({"step": name, "cases": int(keep.sum())})
pd.DataFrame(funnel)

C:\Users\likin\anaconda3\envs\bn4601a\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,step,cases
0,all cases,110182
1,age >= 16,83009
2,presented to the emergency department,13406
3,>= 1 measured vital sign in the text,5690


In [23]:
mc_test = cases.loc[keep, ["case_id", "article_id", "age", "gender", "case_text", "n_vitals", "department_guess"]]
mc_test = mc_test.reset_index(drop=True)
mc_test.to_parquet("reasoning/data/processed/multicare_ed_test.parquet", index=False)
del cases; gc.collect()

print(mc_test["department_guess"].value_counts().to_dict())
print("\nExample case (first 600 characters):\n", mc_test.loc[0, "case_text"][:600])

{'Respiratory': 1527, 'Cardiology': 1160, 'Other/unclear': 994, 'Neurology': 919, 'Surgery': 872, 'Orthopaedics': 218}

Example case (first 600 characters):
 A 29 years old female, physician by profession presented to the emergency department with a history of aggressive vomiting five weeks back followed by left upper abdominal, a single episode of loose motion, subcostal pain radiating to left shoulder associated with shortness of breath (SOB) and was unable to take full inspiration. The patient has a history of heartburn, early satiety, indigestion, and food regurgitation six years ago and diagnosed and managed as gastroesophageal reflux disease in her native country.
The primary evaluation shows a toxic looking afebrile patient with vitals as; r


## Summary: everything 02_datasets produced

In [24]:
rows = []
for f in sorted(Path("reasoning/data/processed").iterdir()):
    n = len(pd.read_parquet(f)) if f.suffix == ".parquet" else sum(1 for _ in open(f, encoding="utf-8"))
    rows.append({"file": f.name, "rows": n, "MB": round(f.stat().st_size / 1e6, 1)})
summary = pd.DataFrame(rows)
summary["used for"] = summary["file"].map({
    "pool1.parquet": "Pool 1: train/test the score heads (KTAS + NHAMCS)",
    "pool2_train.jsonl": "Pool 2: LoRA training",
    "pool2_val.jsonl": "Pool 2: LoRA validation (watch for overfitting)",
    "pool2_test.jsonl": "Pool 2: LoRA test, per source",
    "medqa_bench200.jsonl": "Benchmark: base vs LoRA reasoning",
    "multicare_ed_test.parquet": "Full-pipeline test + demo cases",
})
summary

,file,rows,MB,used for
0,medqa_bench200.jsonl,200,0.2,Benchmark: base vs LoRA reasoning
1,multicare_ed_test.parquet,5690,11.8,Full-pipeline test + demo cases
2,pool1.parquet,9674,0.3,Pool 1: train/test the score heads (KTAS + NHA...
3,pool2_test.jsonl,19324,28.2,"Pool 2: LoRA test, per source"
4,pool2_train.jsonl,347779,505.8,Pool 2: LoRA training
5,pool2_val.jsonl,19322,28.0,Pool 2: LoRA validation (watch for overfitting)
